# Extrapolate Hydrology Model Parameters
This notebook extrapolates the beta parameter to ungaged basin locations.

### Import Libraries

In [0]:
import json
import xarray as xr
import pathlib as pl
import numpy as np
import pickle
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import scipy
from scipy.linalg import LinAlgError
from src.models import extrapolation_funcs as extrap




load data

In [0]:
with open('/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/LWI_output.csv', 'rb') as f:
    LWI_basins = pd.read_csv('/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/LWI_output.csv')

with open('/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/Jacksonville_output.csv', 'rb') as f:
    FL_basins = pd.read_csv('/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/Jacksonville_output.csv')

LWI_FL_basins = pd.concat([LWI_basins, FL_basins], ignore_index=True)
LWI_FL_basins


with open('/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/gage_basin_geometries_extrapolation_soil_smap.pickle', 'rb') as f:
    gage_basins = pickle.load(f)

gage_basins.loc[gage_basins.slope_average == '', 'slope_average'] = np.nan
gage_basins['slope_average'] = gage_basins.slope_average.astype(float)

gage_basins.loc[gage_basins.slope_stddev == '', 'slope_stddev'] = np.nan
gage_basins['slope_stddev'] = gage_basins.slope_stddev.astype(float)

Standardize data

In [0]:
# Ensure 'usgs_site_no' is string and zero-padded in both DataFrames
gage_basins['usgs_site_no'] = gage_basins['usgs_site_no'].astype(str).apply(
    lambda x: '0' + x if not x.startswith('0') else x
)
LWI_FL_basins['usgs_site_no'] = LWI_FL_basins['usgs_site_no'].astype(str).apply(
    lambda x: '0' + x if not x.startswith('0') else x
)

# Select relevant columns from LWI_FL_basins and rename 'beta' to 'Beta'
lwi_replacement = LWI_FL_basins[['usgs_site_no', 'w', 'mu', 'BI', 'beta']].rename(columns={'beta': 'Beta'})

# Merge and update values
new_gage_basins = gage_basins.drop(['w', 'mu', 'BI', 'Beta'], axis=1).merge(
    lwi_replacement, on='usgs_site_no', how='left'
)

new_gage_basins

In [0]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ExpSineSquared, DotProduct, ConstantKernel, RationalQuadratic, Exponentiation, Sum, Product, CompoundKernel, Hyperparameter

relevant_data = new_gage_basins[['slope_average', 'avg_roughness', 'ssurgo_min', 'ssurgo_max', 'ssurgo_wtd_avg', 'DI_mean', 'pi2', 'slope_stddev', 'avg_porosity', 'avg_thickness', 'avg_storage','profile_raw','profile_rel','rootzone_raw','rootzone_rel','surface_raw','surface_rel','mu']].dropna()

new_gage_basins_w_id=new_gage_basins[['usgs_site_no','slope_average', 'avg_roughness', 'ssurgo_min', 'ssurgo_max', 'ssurgo_wtd_avg', 'DI_mean', 'pi2', 'slope_stddev', 'avg_porosity', 'avg_thickness', 'avg_storage','profile_raw','profile_rel','rootzone_raw','rootzone_rel','surface_raw','surface_rel','BI']].dropna()

predictors = ['slope_average', 'avg_roughness', 'ssurgo_min', 'ssurgo_max', 'ssurgo_wtd_avg', 'DI_mean', 'pi2', 'slope_stddev', 'avg_porosity', 'avg_thickness', 'avg_storage','profile_raw','profile_rel','rootzone_raw','rootzone_rel','surface_raw','surface_rel']

x = relevant_data[predictors]
relevant_data.reset_index(drop=True, inplace=True)

#so one of the mu values is a string with a comma at the end for some reason, which doesn't nicely convert to float without removing the comma
for i in range(len(relevant_data['mu'])):
    if type(relevant_data.loc[i, 'mu'])==str:
        if ',' in relevant_data.loc[i, 'mu']:
            relevant_data.loc[i,'mu'] = relevant_data.loc[i,'mu'].replace(',', '')

relevant_data['mu'] = relevant_data['mu'].astype(float)

y = relevant_data['mu']
y.values

scaler = StandardScaler()
scaler.fit(x)

x_scaled = scaler.transform(x)

kf = KFold(n_splits = relevant_data.shape[0], shuffle = True, random_state = 1234)

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import RepeatedKFold, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
# Bin target into quantiles for stratification
y_binned = pd.qcut(y, q=5, labels=False)

# Manually create repeated stratified k-fold
def get_repeated_stratified_kfold(X, y_binned, n_splits=5, n_repeats=10, random_state=1234):
    """Generate train/test indices for repeated stratified k-fold"""
    cv_splits = []
    for repeat in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, 
                             random_state=random_state + repeat)
        for train_idx, test_idx in skf.split(X, y_binned):
            cv_splits.append((train_idx, test_idx))
    return cv_splits

from sklearn.metrics import make_scorer
import numpy as np

# Custom scorer that works in original space
def rmse_original_space(y_log_true, y_log_pred):
    """
    Calculate RMSE in original space when model predicts in log-space
    """
    y_true = np.exp(y_log_true)
    y_pred = np.exp(y_log_pred)
    return np.sqrt(np.mean((y_pred - y_true)**2))

# Create scorer (note: greater_is_better=False for RMSE)
rmse_scorer = make_scorer(rmse_original_space, greater_is_better=False)

y_log = np.log(y + 1e-6)


# ============================================================================
# STEP 1: FEATURE SELECTION
# ============================================================================
print("="*60)
print("STEP 1: FEATURE SELECTION")
print("="*60)

#rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=1234)

# Quick baseline model for feature importance
baseline_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C= 1e-1, gamma=0.05))
])
baseline_pipe.fit(x, y)

# Compute permutation importance
perm_imp = permutation_importance(baseline_pipe, x, y, n_repeats=10, 
                                  random_state=1234, scoring='neg_root_mean_squared_error')

importance_df = pd.DataFrame({
    'feature': predictors,
    'importance': perm_imp.importances_mean,
    'std': perm_imp.importances_std
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importance_df)

# Keep features with positive importance
important_features = importance_df[importance_df['importance'] > 0]['feature'].tolist()
print(f"\n✓ Keeping {len(important_features)}/{len(predictors)} features")
print(f"Selected features: {important_features}")

# Create reduced feature set
x_selected = x[important_features].copy()

# Create the CV splits
rkf = get_repeated_stratified_kfold(x_selected, y_binned, n_splits=5, n_repeats=10, random_state=1234)

# ============================================================================
# STEP 2: TRAIN INDIVIDUAL MODELS
# ============================================================================
print("\n" + "="*60)
print("STEP 2: TRAINING INDIVIDUAL MODELS")
print("="*60)

models = {}

# --- SVR ---
print("\nTraining SVR...")
svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])
svr_params = {
    'svr__C': np.logspace(-2, 2, 5).tolist(),
    'svr__gamma':  np.logspace(-3, -1, 12),
    'svr__kernel': ['linear', 'sigmoid', 'rbf', 'poly'],
    'svr__coef0': [0, 0.5, 1],
    'svr__degree': [1, 2, 3, 4]
}
svr_gs = GridSearchCV(svr_pipe, svr_params, scoring='neg_root_mean_squared_error',
                      cv=rkf, n_jobs=14, refit=True)
svr_gs.fit(x_selected, y_log)
models['SVR'] = svr_gs.best_estimator_
print(f"SVR Best Score: {-svr_gs.best_score_:.2f}")
print(f"SVR Best Params: {svr_gs.best_params_}")

# --- Kernel Ridge ---
print("\nTraining Kernel Ridge...")
kr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('kr', KernelRidge())
])
kr_params = {
    'kr__alpha': np.logspace(-5, 2, 7),
    'kr__gamma': np.logspace(-4, -.5, 10),
    'kr__kernel': ['rbf', 'poly', 'linear', 'sigmoid'],
    'kr__degree': [1, 2, 3],
    'kr__coef0': [0, 0.5, 1]
}
kr_gs = GridSearchCV(kr_pipe, kr_params, scoring='neg_root_mean_squared_error',
                     cv=rkf, n_jobs=14, refit=True)
kr_gs.fit(x_selected, y_log)
models['KernelRidge'] = kr_gs.best_estimator_
print(f"Kernel Ridge Best Score: {-kr_gs.best_score_:.4f}")
print(f"Kernel Ridge Best Params: {kr_gs.best_params_}")

# --- KNN ---
print("\nTraining KNN...")
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])
knn_params = {
    'knn__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [0.1, 0.2, 0.3, 0.4]
}
knn_gs = GridSearchCV(knn_pipe, knn_params, scoring='neg_root_mean_squared_error',
                      cv=rkf, n_jobs=14, refit=True)
knn_gs.fit(x_selected, y_log)
models['KNN'] = knn_gs.best_estimator_
print(f"KNN Best Score: {-knn_gs.best_score_:.4f}")
print(f"KNN Best Params: {knn_gs.best_params_}")

# --- Random Forest ---
print("\nTraining Random Forest...")
rf_pipe = Pipeline([
    ('rf', RandomForestRegressor(random_state=1234))
])
rf_params = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [5, 10, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}
rf_gs = GridSearchCV(rf_pipe, rf_params, scoring='neg_root_mean_squared_error',
                     cv=rkf, n_jobs=14, refit=True)
rf_gs.fit(x_selected, y_log)
models['RandomForest'] = rf_gs.best_estimator_
print(f"RF Best Score: {-rf_gs.best_score_:.2f}")
print(f"RF Best Params: {rf_gs.best_params_}")

# --- Gradient Boosting ---
print("\nTraining Gradient Boosting...")
gb_pipe = Pipeline([
    ('gb', GradientBoostingRegressor(random_state=1234))
])
gb_params = {
    'gb__n_estimators': [50, 100, 200],
    'gb__learning_rate': [0.01, 0.05, 0.1],
    'gb__max_depth': [3, 5, 7],
    'gb__subsample': [0.8, 1.0]
}
gb_gs = GridSearchCV(gb_pipe, gb_params, scoring='neg_root_mean_squared_error',
                     cv=rkf, n_jobs=14, refit=True)
gb_gs.fit(x_selected, y_log)
models['GradientBoosting'] = gb_gs.best_estimator_
print(f"GB Best Score: {-gb_gs.best_score_:.2f}")
print(f"GB Best Params: {gb_gs.best_params_}")

# ============================================================================
# STEP 3: ENSEMBLE - STACKING
# ============================================================================
print("\n" + "="*60)
print("STEP 3: ENSEMBLE STACKING")
print("="*60)

stacker = StackingRegressor(
    estimators=[
        ('svr', svr_gs.best_estimator_),
        ('rf', rf_gs.best_estimator_),
        ('gb', gb_gs.best_estimator_)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5  # Internal CV for meta-learner
)

print("\nTraining stacked ensemble...")
stacker.fit(x_selected, y_log)
models['StackedEnsemble'] = stacker

# ============================================================================
# STEP 4: COMPARE ALL MODELS
# ============================================================================
print("\n" + "="*60)
print("STEP 4: MODEL COMPARISON")
print("="*60)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, x_selected, y_log, cv=rkf, 
                             scoring=rmse_scorer, n_jobs=12) #scoring='neg_root_mean_squared_error',
    rmse_mean = -scores.mean()
    rmse_std = scores.std()
    results[name] = {'mean': rmse_mean, 'std': rmse_std}
    print(f"{name:20s}: RMSE = {rmse_mean:.2f} ± {rmse_std:.2f}")

# Find best model
best_model_name = min(results, key=lambda k: results[k]['mean'])
best_model = models[best_model_name]

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   RMSE = {results[best_model_name]['mean']:.2f} ± {results[best_model_name]['std']:.2f}")

# ============================================================================
# STEP 5: DETAILED EVALUATION OF BEST MODEL
# ============================================================================
print("\n" + "="*60)
print("STEP 5: DETAILED EVALUATION")
print("="*60)

print(f"\nBest Model ({best_model_name}) Performance:")
print(f"  Repeated CV RMSE: {results[best_model_name]['mean']:.2f} ± {results[best_model_name]['std']:.2f}")
print(f"  Target Std: {y.std():.2f}")
print(f"  RMSE/Std: {results[best_model_name]['mean']/y.std():.2f}")

# Train final model on all data for predictions
print(f"\nTraining final {best_model_name} on all data...")
#best_model.fit(x_selected, y)
#y_pred_final = best_model.predict(x_selected)

best_model.fit(x_selected, y_log)
y_pred_log_final = best_model.predict(x_selected)
y_pred_final = np.exp(y_pred_log_final)  # Back-transform to original scale


# These are training predictions (optimistic), not CV
rmse_train = np.sqrt(mean_squared_error(y, y_pred_final))
mae_train = mean_absolute_error(y, y_pred_final)
print(f"  Training RMSE: {rmse_train:.2f} (optimistic)")
print(f"  Training MAE:  {mae_train:.2f}")

# Visualize
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs Actual
axes[0].scatter(y_pred_final, y, alpha=0.6)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'{best_model_name}: Training RMSE={rmse_train:.1f}')  # Fixed!

# Residuals
residuals = y - y_pred_final
axes[1].scatter(y_pred_final, residuals, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

# Residual distribution
axes[2].hist(residuals, bins=20, edgecolor='black')
axes[2].axvline(0, color='r', linestyle='--', lw=2)
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 6: EXPORT PREDICTIONS
# ============================================================================
print("\n" + "="*60)
print("STEP 6: EXPORTING PREDICTIONS")
print("="*60)

export_df = new_gage_basins_w_id[['usgs_site_no']].copy()
export_df['actual_mu'] = y.values
export_df['predicted_mu'] = y_pred_final
export_df['residual'] = residuals
export_df['abs_error'] = np.abs(residuals)
export_df['pct_error'] = 100 * residuals / y.values

# Add selected features
for col in important_features:
    export_df[col] = x_selected[col].values

export_df2=LWI_FL_basins[['usgs_site_no']].merge(export_df, on='usgs_site_no', how='left')
export_df2.to_csv('mu_predictions_with_feature_selection_and_ensemble.csv', index=False)
print(f"✓ Exported {len(export_df)} predictions to CSV")
print(f"\nTop 5 predictions:")
print(export_df[['usgs_site_no', 'actual_mu', 'predicted_mu', 'abs_error']].head())

# ============================================================================
# STEP 7: FEATURE IMPORTANCE ANALYSIS & EXPORT
# ============================================================================
print("\n" + "="*60)
print("STEP 7: FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Calculate permutation importance on the best model
print(f"\nCalculating permutation importance for {best_model_name}...")
perm_importance = permutation_importance(
    best_model,  # Already fitted in Step 5
    x_selected, 
    y_log,
    n_repeats=10,
    random_state=1234,
    scoring='neg_root_mean_squared_error'
)

# Create a dataframe to view results
importance_df = pd.DataFrame({
    'feature': important_features,  # Use the selected features
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("\nFeature Importance (Permutation):")
print(importance_df)

# Visualize
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], 
         xerr=importance_df['std'], alpha=0.7)
plt.xlabel('Permutation Importance (RMSE increase when shuffled)')
plt.ylabel('Feature')
plt.title(f'Feature Importance for {best_model_name} Model')
plt.gca().invert_yaxis()
plt.axvline(0, color='red', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

# Export to CSV
importance_path = '/dbfs/mnt/lwi-transition-zone/data/hydrology/baseflow/mu_feature_importance.csv'
importance_df.to_csv(importance_path, index=False)
print(f"\n✓ Feature importance exported to: {importance_path}")

# Summary statistics
print(f"\nImportance Summary:")
print(f"  Most important feature: {importance_df.iloc[0]['feature']} "
      f"(importance: {importance_df.iloc[0]['importance']:.4f})")
print(f"  Least important feature: {importance_df.iloc[-1]['feature']} "
      f"(importance: {importance_df.iloc[-1]['importance']:.4f})")
print(f"  Mean importance: {importance_df['importance'].mean():.4f}")
print(f"  Features with negative importance: {(importance_df['importance'] < 0).sum()}")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

summary_df = pd.DataFrame(results).T
summary_df.columns = ['RMSE_mean', 'RMSE_std']
summary_df = summary_df.sort_values('RMSE_mean')
summary_df['improvement_vs_worst'] = (summary_df['RMSE_mean'].max() - summary_df['RMSE_mean']) / summary_df['RMSE_mean'].max() * 100

print("\nModel Performance Summary:")
print(summary_df.to_string())
print(f"\nBest improvement: {summary_df['improvement_vs_worst'].max():.1f}% vs worst model")

Here we try a gridsearch for the best model using all predictors and a range of parameterizations for each of several model families and show their performance and best parameterizations

In [0]:
export_df

In [0]:



#standardize data

svr = GridSearchCV(
    SVR(kernel="rbf", gamma=0.1),
    param_grid={"C":[ 1e-1, 1, 1e-2], "gamma": np.logspace(-3, -1, 10), 'kernel': [ 'linear', 'sigmoid', 'rbf', 'poly'], 'coef0': [0.6,  0.5, 0.4, 0.3, 0.2], 'degree': [1, 2, 3, 4]},
    scoring = 'neg_root_mean_squared_error', cv = kf
)
svr.fit(x_scaled,y)

kr = GridSearchCV(
    KernelRidge(kernel="rbf", gamma=0.1),
    param_grid={"alpha": [10, 1e0, 100 ], "gamma": np.logspace(-3, -1, 10), 'kernel': ['additive_chi2', 'chi2', 'linear', 'poly', 'polynomial', 'rbf', 'laplacian', 'sigmoid', 'cosine'], 'degree': [1,2,3], 'coef0': [ 0.3, 0.2, 0.1]},
    scoring = 'neg_root_mean_squared_error', cv = kf
)
kr.fit(x_scaled, y)

knn = GridSearchCV(
    KNeighborsRegressor(),
    param_grid = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 'weights': ['uniform', 'distance'], 'p': [0.1, 0.2, 0.4, 0.3,]},  n_jobs = 4,
    scoring = 'neg_root_mean_squared_error', cv = kf
)
knn.fit(x_scaled, y)

rn = GridSearchCV(
    RadiusNeighborsRegressor(),
    param_grid = {'radius': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2], 'weights':['uniform', 'distance'],
                  'p':[0.1, 0.5, 1, 1.5, 2]},  n_jobs = 4, scoring = 'neg_root_mean_squared_error', cv = kf
    )
rn.fit(x_scaled, y)

gp = GridSearchCV(
    GaussianProcessRegressor(),
    param_grid = {'alpha': [1000, 100, 10, 1e0], 'kernel': [RBF(), RationalQuadratic(), ExpSineSquared(), Matern(), WhiteKernel(), RBF(length_scale = 1.1), RBF(length_scale = 0.9), None, RBF() + WhiteKernel(noise_level = .0001) ]},
    n_jobs = 4, scoring = 'neg_root_mean_squared_error', cv = kf
    )
gp.fit(x_scaled, y)

'''
this_kr = KernelRidge(kernel="rbf", gamma=0.1, alpha = 1e-3)
this_kr.fit(x,y)'''


print(f'svr:{svr.best_score_}, params:{svr.best_params_}')
print(f'knn:{knn.best_score_}, params:{knn.best_params_}')
print(f'gp:{gp.best_score_}, params:{gp.best_params_}')
print(f'kr:{kr.best_score_}, params:{kr.best_params_}')





#we get a lot of heteroskedasticity where y's variance scales with its value / predicted value. So lets try logging y and see what happens



In [0]:
print(f'svr:{svr.best_score_}, params:{svr.best_params_}')
print(f'knn:{knn.best_score_}, params:{knn.best_params_}')
print(f'gp:{gp.best_score_}, params:{gp.best_params_}')
print(f'kr:{kr.best_score_}, params:{kr.best_params_}')




rmse should be interpreted in the context of the standard deviation of the response

In [0]:
print(y.std())

Now we plot the error as a function of predicted value - this helps us understand to what extent the performance we see represents a real trend, and also helps us to see whether they are obviously overfitting

In [0]:
plt.scatter(svr.predict(x_scaled), y)#maybe cone trend

In [0]:
plt.scatter(kr.predict(x_scaled), y) #maybe cone trend

In [0]:

plt.scatter(knn.predict(x_scaled), y) #cone trend

In [0]:
plt.scatter(gp.predict(x_scaled), y) #strong cone trend

non-loggged kr is the best-performing model as evaluated by pseudo-R^2.

Here we iteratively remove and add predictors to the best-performing model until we've identified the predictor set that maximizes model performance, using the best model family and parameterization as identified above

In [0]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

#Okay so the kr model is clearly the best performer but it trains so slowly that the recursive feature selection is taking an unreasonably long time - like 16 hours and we don't even get the deterministic from-full and from-null versions. The gaussian process is second best, so let's try feature selection with that and see how kr does with the resulting feature set. 


candidate_predictors = ['slope_average', 'avg_roughness', 'ssurgo_min', 'ssurgo_max', 'ssurgo_wtd_avg', 'DI_mean', 'pi2', 'slope_stddev', 'avg_porosity', 'avg_thickness', 'avg_storage','profile_raw','profile_rel','rootzone_raw','rootzone_rel','surface_raw','surface_rel']

x = relevant_data[candidate_predictors]

x_scaled_candidates = scaler.transform(x)

chosen_predictors = []

#best_model.fit(x_scaled, np.log(y), scoring = 'neg_root_mean_squared_error',cv = kf)
kr = GridSearchCV(
    KernelRidge(),
    param_grid={'alpha': [10], 'coef0': [0.2], 'degree': [2], 'gamma':[0.01291549665014884], 'kernel': ['poly']},
    scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4
)
'''
gp = GridSearchCV(GaussianProcessRegressor(),
                  param_grid = {'alpha': [100], 'kernel': [RBF(length_scale=1)]},
                  scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4)'''
#best_model.score_

optimal_preds, score = extrap.optimize_model_rand_restarts(kr, x_scaled_candidates, y, kf, range(len(candidate_predictors)), n_restarts = 10)

In [0]:
print(optimal_preds)

In [0]:
print(score)

now let's just do a little post-hoc parameter tuning (expanding and contracting my grisearch by hand in a hill-climbing fashion because a full parameter grid is frustratingly slow to evaluate)

In [0]:

optimal_preds = [1, 2, 12, 11, 9, 0, 6]
optimal_xscaled = x_scaled[:, optimal_preds]

kr = GridSearchCV(
    KernelRidge(),
    param_grid={"alpha": [10, 1e0, 100 ], "gamma": np.logspace(-3, -1, 10), 'kernel': ['additive_chi2', 'chi2', 'linear', 'poly', 'polynomial', 'rbf', 'laplacian', 'sigmoid', 'cosine'], 'degree': [1,2,3,4], 'coef0': [ 0.3, 0.2, 0.1]},
    scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4
)

kr.fit(optimal_xscaled, y)

kr.best_score_



In [0]:

with open('/dbfs/mnt/lwi-transition-zone/data/hydrology/extrapolation/final_model_mu.pkl', 'wb') as f:
    pickle.dump(kr, f)

Some of the models above show heteroskedasticity, with greater error for greater predicted values. This is often handled by logging the response variable, so we repeat the process with the response variable logged.

In [0]:


#standardize data
#this one takes 26 minutes to run
svr_log = GridSearchCV(
    SVR(kernel="rbf", gamma=0.1),
    param_grid={"C":[  0.4, 0.3, 0.2, 0.1, 0.01], "gamma": np.logspace(-2, 0, 10), 'kernel': ['additive_chi2', 'chi2', 'linear', 'poly', 'polynomial', 'rbf', 'laplacian', 'sigmoid', 'cosine'], 'coef0': [0.5, 0.4, 0.6, 0.7, 0.8, 0.9, 1.], 'degree': [1, 2, 3, 4]},
    scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4
)
svr_log.fit(x_scaled,np.log(y))

#running
kr_log = GridSearchCV(
    KernelRidge(kernel="rbf", gamma=0.1),
    param_grid={"alpha": [1e0, 0.1, .01, .001 ], "gamma": np.logspace(-4, -2, 3), 'kernel': ['additive_chi2', 'chi2', 'linear', 'poly', 'polynomial', 'rbf', 'laplacian', 'sigmoid', 'cosine'], 'degree': [1,2,3, 4], 'coef0': [ 0.4, 0.,5, 0.6, 0.7,0.3, 0.2, 0.1]},
    scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4
)
kr_log.fit(x_scaled, np.log(y))
#running
knn_log = GridSearchCV(
    KNeighborsRegressor(),
    param_grid = {'n_neighbors': [ 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 'weights': ['uniform', 'distance'], 'p': [0.1, 0.2, 0.4, 0.3]},
    scoring = 'neg_root_mean_squared_error', cv = kf, n_jobs = 4)
knn_log.fit(x_scaled, np.log(y))



gp_log = GridSearchCV(
    GaussianProcessRegressor(),
    param_grid = {'alpha': [ 10, 1e0, 1e-1, 1e-2], 'kernel': [RBF(), RationalQuadratic(), RationalQuadratic(length_scale = 1.2, alpha = 0.8),  RationalQuadratic(length_scale = 1.2, alpha = 0.9) + WhiteKernel(noise_level=.1),  RationalQuadratic(length_scale = 1.2, alpha = 0.9) + WhiteKernel(noise_level=.01), RationalQuadratic(length_scale = 1.2, alpha = 0.9) + WhiteKernel(noise_level=.001), RationalQuadratic(length_scale = 1.2, alpha = 0.9), ExpSineSquared(), Matern(), WhiteKernel(), None]},
    n_jobs = 4, scoring = 'neg_root_mean_squared_error', cv = kf
    )
gp_log.fit(x_scaled, np.log(y))

'''
this_kr = KernelRidge(kernel="rbf", gamma=0.1, alpha = 1e-3)
this_kr.fit(x,y)'''
svr_log.best_score_ #1.18, 
kr_log.best_score_ #1.21, 
knn_log.best_score_ #1.18,
gp_log.best_score_ #1.27, 

svr_log.best_params_
kr_log.best_params_
knn_log.best_params_
gp_log.best_params_


print(f'svr:{svr_log.best_score_}, params:{svr_log.best_params_}')
print(f'knn:{knn_log.best_score_}, params:{knn_log.best_params_}')
print(f'gp:{gp_log.best_score_}, params:{gp_log.best_params_}')
print(f'kr:{kr_log.best_score_}, params:{kr_log.best_params_}')


#we get a lot of heteroskedasticity where y's variance scales with its value / predicted value. So lets try logging y and see what happens

np.log(y).std() #1.6

In [0]:
print(f'svr:{svr_log.best_score_}, params:{svr_log.best_params_}')
print(f'knn:{knn_log.best_score_}, params:{knn_log.best_params_}')
print(f'gp:{gp_log.best_score_}, params:{gp_log.best_params_}')
print(f'kr:{kr_log.best_score_}, params:{kr_log.best_params_}')



Now we again plot the error as a function of predicted value - this helps us understand to what extent the performance we see represents a real trend, and also helps us to see whether they are obviously overfitting

In [0]:
plt.scatter(svr_log.predict(x_scaled), np.log(y))#weak trend

In [0]:
plt.scatter(knn_log.predict(x_scaled), np.log(y)) #weak trend

In [0]:
plt.scatter(kr_log.predict(x_scaled), np.log(y)) #weak trend

In [0]:
plt.scatter(gp_log.predict(x_scaled), np.log(y)) #clear overfit

None of the logged models performed well enough to be worth continuing to refine, so we stop here